# Chapter 10 — Rough, Temporal, and Fuzzy Modelling
### Notebook 2 · Vagueness and granularity

*Book reference: Section 10.2*

Two different failures of crispness. **Vagueness**: there is no height at which 'tall' switches on. **Granularity**: your attributes cannot tell two objects apart, so some sets are not exactly describable at all. Different problems, different formalisms.

In [ ]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

In [ ]:
import sys, json, logging
from pathlib import Path
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate)); break
sys.path.insert(0, str(Path.cwd()))

import ch10_toolkit as ch10
import pandas as pd
logging.getLogger("dspy").setLevel(logging.WARNING)

## 1. Vagueness: fuzzy membership

A crisp definition of `tall` needs a threshold, and every threshold is arbitrary at exactly the point where the decision is hard. Fuzzy sets replace the boundary with a **degree**.

In [ ]:
heights = [150, 160, 165, 170, 175, 180, 185, 190, 200]
rows = [{'height': h, **{name: ch10.membership(name, h)
                         for name in ch10.FUZZY_SETS}} for h in heights]
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
print('A crisp threshold at 180cm:')
for h in [179, 180, 181]:
    print(f'  {h}cm -> tall={h >= 180}   (fuzzy: {ch10.membership("tall", h)})')
print('\nThe crisp version says two people 2cm apart are in different\n'
      'categories, and two people 20cm apart (181 and 201) are in the same\n'
      'one. The fuzzy version says what everyone actually means.')

### Combining degrees

A **t-norm** generalises conjunction. Two common choices disagree, and the choice is a modelling decision rather than a detail.

In [ ]:
print(f"{'x':>5s}{'y':>6s}{'min':>8s}{'product':>10s}")
for x, y in [(1.0, 0.5), (0.8, 0.8), (0.5, 0.5), (0.2, 0.9)]:
    print(f'{x:5.1f}{y:6.1f}{ch10.fuzzy_and(x, y):8.2f}'
          f'{ch10.fuzzy_and(x, y, "product"):10.2f}')
print('\nmin keeps the worst input; product punishes every additional\n'
      'condition. Neither is right in general -- but they give different\n'
      'answers, so the choice has to be made deliberately.')

In [ ]:
values = list(range(140, 221))
print('degree to which every tall person is average:',
      ch10.subsumption_degree('tall', 'average', values))
print('degree to which every tall person is tall   :',
      ch10.subsumption_degree('tall', 'tall', values))
print('\nFuzzy subsumption takes the INFIMUM over the domain: the worst point\n'
      'decides. One tall-but-not-average height is enough to drive the degree\n'
      'to zero -- the fuzzy reading of "every" is as demanding as the crisp one.')

### Alpha-cuts: back to crisp, deliberately

Sooner or later something must act. An **alpha-cut** turns a fuzzy set back into a crisp one — and reintroduces exactly the arbitrary threshold fuzzification was meant to avoid. The gain is that the choice is now *explicit*.

In [ ]:
for alpha in [0.25, 0.5, 0.75, 1.0]:
    cut = ch10.alpha_cut('tall', alpha, values)
    print(f'  alpha={alpha}: tall means >= {min(cut)}cm  ({len(cut)} heights)')
print('\nThe threshold did not disappear; it moved somewhere it can be argued\n'
      'about. That is a real improvement over a magic number in a SPARQL\n'
      'filter, and it is the whole practical benefit.')

## 2. Granularity: rough sets

A different problem entirely. Here nothing is vague — the trouble is that your **attributes cannot separate** some objects.

In [ ]:
system = ch10.SAMPLE_SYSTEM
rows = [{'patient': name, **values} for name, values in system.objects.items()]
print(pd.DataFrame(rows).to_string(index=False))
print()
print('indiscernibility classes (granules):')
for group in system.indiscernibility():
    print('  ', group)

`p1` and `p2` are identical on every recorded attribute, as are `p3` and `p4`. No reasoning can separate them — so any set that contains one but not the other is **not exactly describable**.

In [ ]:
target = ['p1', 'p3', 'p5']
print('target set        :', target)
print('lower approximation:', system.lower_approximation(target),
      ' (certainly in)')
print('upper approximation:', system.upper_approximation(target),
      ' (possibly in)')
print('boundary region    :', system.boundary(target),
      ' (cannot decide)')
print('accuracy           :', system.accuracy(target))

In [ ]:
exact = ['p1', 'p2', 'p5']
print('a set that respects the granules:', exact)
print('  lower   :', system.lower_approximation(exact))
print('  upper   :', system.upper_approximation(exact))
print('  boundary:', system.boundary(exact) or '(empty)')
print('  accuracy:', system.accuracy(exact))
assert system.accuracy(exact) == 1.0
print('\nAccuracy 1.0: this set IS exactly describable with the attributes you\n'
      'have. Rough set theory does not add uncertainty -- it MEASURES the\n'
      'uncertainty your data already had.')

> **The distinction worth keeping.** Fuzzy handles predicates with no sharp boundary. Rough handles data too coarse to draw a boundary you already believe in. Reaching for the wrong one produces a model that answers a question nobody asked.

### Exercise 2.1 — Show that adding an attribute shrinks the boundary

Add a third attribute that separates `p3` from `p4`, and show the accuracy of a target set improving.

> **Hint.** Give `p3` and `p4` different values for the new attribute.

In [ ]:
# YOUR CODE HERE


<details>
<summary>Solution 2.1</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [ ]:
richer = ch10.InformationSystem({
    name: dict(values) for name, values in ch10.SAMPLE_SYSTEM.objects.items()})
for name, age in [('p1', 'young'), ('p2', 'young'), ('p3', 'young'),
                  ('p4', 'old'), ('p5', 'old'), ('p6', 'young')]:
    richer.objects[name]['age'] = age

target = ['p1', 'p3', 'p5']
before = ch10.SAMPLE_SYSTEM.accuracy(target)
after = richer.accuracy(target)
print('granules before:', ch10.SAMPLE_SYSTEM.indiscernibility())
print('granules after :', richer.indiscernibility())
print(f'\naccuracy {before} -> {after}')
print('boundary before:', ch10.SAMPLE_SYSTEM.boundary(target))
print('boundary after :', richer.boundary(target))
assert after > before
print('\nOne more attribute split a granule and moved two patients out of the\n'
      'boundary region. Rough set accuracy therefore measures something\n'
      'actionable: it tells you whether collecting another field would help.')

### Exercise 2.2 — Pick the right formalism for three requirements

For each requirement, say whether it needs fuzzy, rough or crisp modelling, and defend the choice in one sentence.

In [ ]:
# YOUR CODE HERE


<details>
<summary>Solution 2.2</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [ ]:
cases = {
    'Alert when the patient has a high fever.': 'fuzzy',
    'Two records agreeing on every field must be one case.': 'rough',
    'Each admission has exactly one identifier.': 'crisp',
}
for text, expected in cases.items():
    print(f'{expected:8s} <- {text}')
print()
print('high fever   : "high" has no sharp cut-off; a threshold would be')
print('               arbitrary at exactly the temperatures clinicians argue about.')
print('identical rec: nothing is vague -- the ATTRIBUTES are too coarse, which is')
print('               the rough-set situation, not the fuzzy one.')
print('identifier   : sharp by construction; extra machinery buys nothing and')
print('               costs reasoning time.')
assert set(cases.values()) == {'fuzzy', 'rough', 'crisp'}